In [1]:
# Standard Libraries
import os
import numpy as np
import pandas as pd

# PyTorch & PyTorch Geometric
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv

# RDKit for Molecular Processing
from rdkit import Chem
from rdkit.Chem import AllChem

# Visualization & Debugging
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader  # PyG's DataLoader for batching graphs

In [2]:
# --------------------
# ✅ Convert SMILES to Graph Representation
# --------------------

def molecule_to_graph(smiles):
    """
    Converts a SMILES string into a graph representation.
    
    Returns:
    - PyG Data object with atom and bond information, or None if invalid.
    """
    mol = Chem.MolFromSmiles(smiles)

    # Handle invalid SMILES
    if mol is None:
        print(f"⚠️ Warning: Invalid SMILES '{smiles}' was skipped.")
        return None

    # Extract atom types
    atom_types = [atom.GetAtomicNum() for atom in mol.GetAtoms()]

    # Extract bonds
    edge_index = []
    bond_types = []

    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])  # Ensure undirected edges
        bond_types.append(float(bond.GetBondTypeAsDouble()))  # Ensure float type
        bond_types.append(float(bond.GetBondTypeAsDouble()))

    # Convert to tensors
    x = torch.tensor(atom_types, dtype=torch.float32).view(-1, 1)  # Convert to float32
    edge_index = torch.tensor(edge_index, dtype=torch.long).T if edge_index else torch.empty((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(bond_types, dtype=torch.float32).view(-1, 1) if bond_types else torch.empty((0, 1), dtype=torch.float32)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

In [3]:
# --------------------
# ✅ Load Dataset as Graphs
# --------------------
# Load dataset
file_path = "/home/jovyan/Research Project/sdf_data/data.pkl"
df = pd.read_pickle(file_path)

# Extract SMILES list
smiles_list = df["SMILES"].dropna().tolist()

# Convert SMILES to Graphs
dataset = [molecule_to_graph(smi) for smi in smiles_list]
dataset = [graph for graph in dataset if graph is not None]  # Remove invalid molecules

# Create DataLoader
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)


print(f"✅ Dataset successfully loaded with {len(dataset)} valid molecular graphs.")

[10:49:58] Explicit valence for atom # 51 C, 5, is greater than permitted
[10:49:58] Explicit valence for atom # 37 C, 5, is greater than permitted
[10:49:59] Explicit valence for atom # 18 Si, 5, is greater than permitted
[10:49:59] Explicit valence for atom # 28 C, 6, is greater than permitted
[10:49:59] Explicit valence for atom # 16 C, 6, is greater than permitted
[10:49:59] Explicit valence for atom # 8 C, 5, is greater than permitted
[10:49:59] Explicit valence for atom # 8 C, 5, is greater than permitted
[10:49:59] Explicit valence for atom # 3 C, 5, is greater than permitted
[10:49:59] Explicit valence for atom # 3 C, 5, is greater than permitted
[10:49:59] Explicit valence for atom # 3 C, 5, is greater than permitted


⚠️ Warning: Invalid SMILES '[H]C1=C([H])N2B(N1c1c(C([H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c([H])c1C([H])(C([H])([H])[H])C([H])([H])[H])O1->[InH]O3B4N(c5c(C([H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c([H])c5C([H])(C([H])([H])[H])C([H])([H])[H])C([H])=C([H])N4c45c6(C([H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c([H])c4(C([H])(C([H])([H])[H])C([H])([H])[H])~[K]<-31~6~54789%10~C1([H])=C~4([H])C~7(C([H])(C([H])([H])[H])C([H])([H])[H])=C2~8C~9(C([H])(C([H])([H])[H])C([H])([H])[H])=C1~%10[H]' was skipped.
⚠️ Warning: Invalid SMILES '[H]c1c([H])c(C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H])c([H])c2c1-c1c([H])c([H])c(C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H])c([H])c1[B-]21[N+](C([H])(C([H])([H])[H])C([H])([H])[H])=C2[B@-]3([H])c4c([H])c(C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c4C45=C36C3([H])=C7(C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H])C8([H])=C4([H])~[K+]~8~3~5~6~749(<-O21)<-N(C([H])([H])[H])(C([H])([H])[H])C([H])([H])C([H])([H])N->4(C([H])

[10:49:59] Explicit valence for atom # 13 C, 5, is greater than permitted
[10:49:59] Explicit valence for atom # 47 C, 5, is greater than permitted
[10:49:59] Explicit valence for atom # 9 B, 5, is greater than permitted
[10:49:59] Explicit valence for atom # 10 C, 5, is greater than permitted
[10:49:59] Explicit valence for atom # 10 C, 5, is greater than permitted
[10:49:59] Explicit valence for atom # 10 C, 5, is greater than permitted


⚠️ Warning: Invalid SMILES '[H]c1c([H])c([H])c(B2~N(c3c([H])c([H])c([H])c([H])c3[H])N3C45=C6([H])C7([H])=C8([H])C9([H])=C4([H])~[KH]~8~7~9~6~54%10~c5([H])c([H])c([H])c([H])c~4(c5~%10[H])B~3~C~2(C([H])([H])[H])[K]2345678(<-O9C([H])([H])C([H])([H])C([H])([H])C9([H])[H])(~C~C~2([H])~C~3[H])~C2~C~4([H])~C~5([H])~C~6([H])~C~7([H])~C~2~8[H])c([H])c1[H]' was skipped.
⚠️ Warning: Invalid SMILES '[H]c1c([H])c(C([H])(C([H])([H])[H])C([H])([H])[H])c(N2c3c([H])c(C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H])c([H])c4c3O3->[Th+2]256789%10%11%12%13%14(N(c2c(C([H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c([H])c2C([H])(C([H])([H])[H])C([H])([H])[H])c2c([H])c(C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H])c([H])c(c23)C4(C([H])([H])[H])C([H])([H])[H])(~C2([H])=C~5([H])C~6([H])=C~7(C([H])([H])[B-](c3c(F)c(F)c(F)c(F)c3F)(c3c(F)c(F)c(F)c(F)c3F)c3c(F)c(F)c(F)c(F)c3F)C~8([H])=C2~9[H])~C2([H])=C~%10([H])C~%11([H])=C~%12(C([H])([H])[B-](c3c(F)c(F)c(F)c(F)c3F)(c3c(F)c(F)c(F)c(F)c3F)c3c(F)c(F)c(F)c(F)c3F)C~

[10:49:59] Explicit valence for atom # 37 C, 5, is greater than permitted
[10:49:59] Can't kekulize mol.  Unkekulized atoms: 0 1 2 5 7 8 9 10 23 24 25 26 29 30 31 32 33
[10:49:59] Explicit valence for atom # 11 B, 5, is greater than permitted


⚠️ Warning: Invalid SMILES '[H]C1=C([H])N2B(N1c1c(C([H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c([H])c1C([H])(C([H])([H])[H])C([H])([H])[H])O1->[AlH]O3B4N(c5c(C([H])(C([H])([H])[H])C([H])([H])[H])c([H])c([H])c([H])c5C([H])(C([H])([H])[H])C([H])([H])[H])C([H])=C([H])N4C45=C6(C([H])(C([H])([H])[H])C([H])([H])[H])C7([H])=C8([H])C9([H])=C4(C([H])(C([H])([H])[H])C([H])([H])[H])~[K]<-31~8~7~9~5~64%10%11%12%13~C1([H])=C~4([H])C~%10(C([H])(C([H])([H])[H])C([H])([H])[H])=C2~%11C~%12(C([H])(C([H])([H])[H])C([H])([H])[H])=C1~%13[H]' was skipped.
⚠️ Warning: Invalid SMILES '[H]c1c([H])c([H])c23c4(c1[H])-c15c([H])c([H])c([H])c([H])c16C21B(N(C([H])([H])[H])C([H])([H])[H])B(N(C([H])([H])[H])C([H])([H])[H])C27c89c([H])c([H])c([H])c([H])c8%10-c8%11c([H])c([H])c([H])c([H])c82~[Ca]~4~5~%10~%11~3~6~9~1~7(<-O1C([H])([H])C([H])([H])C([H])([H])C1([H])[H])<-O1C([H])([H])C([H])([H])C([H])([H])C1([H])[H]' was skipped.
⚠️ Warning: Invalid SMILES '[H]c1c([H])c([H])c([N+](c2c([H])c([H])c([H])c([H])c2[H])=[B-]2

[10:49:59] Explicit valence for atom # 17 C, 5, is greater than permitted


In [4]:
# --------------------
# ✅ Define Graph-Based Generator and Discriminator
# --------------------
class GraphGenerator(nn.Module):
    def __init__(self, latent_dim, node_dim):
        super(GraphGenerator, self).__init__()
        self.conv1 = GCNConv(latent_dim, 64)
        self.conv2 = GCNConv(64, node_dim)
        self.activation = nn.ReLU()

    def forward(self, z, edge_index):
        x = self.activation(self.conv1(z, edge_index))
        x = self.conv2(x, edge_index)
        return x  # Output node types

In [5]:
class GraphDiscriminator(nn.Module):
    def __init__(self, node_dim, hidden_dim):
        super(GraphDiscriminator, self).__init__()
        self.conv1 = GCNConv(node_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)  # Ensure correct shape
        self.activation = nn.LeakyReLU(0.2)

    def forward(self, x, edge_index, edge_weight=None):
        # Ensure all inputs are float32
        x = x.float()  
        if edge_weight is not None:
            edge_weight = edge_weight.float()

        x = self.activation(self.conv1(x, edge_index, edge_weight))  # GCN Layer

        # Ensure correct shape before passing to Linear Layer
        if x.dim() == 2:  # Expected: [num_nodes, hidden_dim]
            x = x.mean(dim=0, keepdim=True)  # Aggregate over all nodes
        elif x.dim() == 3:  # If batched, aggregate per graph
            x = x.mean(dim=1)  

        return self.fc(x)  # Ensure correct shape for Linear Layer

In [6]:
# --------------------
# ✅ Initialize Models
# --------------------
generator = GraphGenerator(latent_dim=128, node_dim=1)
discriminator = GraphDiscriminator(node_dim=1, hidden_dim=64)
optimizer_G = optim.Adam(generator.parameters(), lr=0.0001, betas=(0.5, 0.9))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.9))

In [ ]:
# --------------------
# ✅ Training Loop (Graph-Based WGAN-GP)
# --------------------
n_epochs = 1000
for epoch in range(n_epochs):
    for real_graph in dataloader:
        optimizer_D.zero_grad()
        
        # Ensure input tensors are float32
        real_x = real_graph.x.float()
        edge_index = real_graph.edge_index
        edge_attr = real_graph.edge_attr.float() if real_graph.edge_attr is not None else None

        # Generate random latent vectors
        z = torch.randn(real_graph.x.shape[0], 128).float()

        # Generate fake graphs
        fake_x = generator(z, edge_index)

        # Compute discriminator loss
        real_scores = discriminator(real_x, edge_index, edge_attr)  # Now correctly shaped
        fake_scores = discriminator(fake_x.detach(), edge_index, edge_attr)
        loss_D = -torch.mean(real_scores) + torch.mean(fake_scores)
        
        loss_D.backward()
        optimizer_D.step()

        # Train Generator
        optimizer_G.zero_grad()
        fake_scores = discriminator(fake_x, edge_index, edge_attr)
        loss_G = -torch.mean(fake_scores)

        loss_G.backward()
        optimizer_G.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss D: {loss_D.item()}, Loss G: {loss_G.item()}")

Epoch 0, Loss D: -0.3263292908668518, Loss G: -0.10024590790271759
Epoch 500, Loss D: 0.9024529457092285, Loss G: 4.725439071655273
